# Classificação - Treino/Teste

## Probabilísticas

### Regressão Logística

#### Treinando e Salvando Modelo

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import joblib

df = pd.read_csv("comentarios.csv")
df.columns = [c.strip().lower() for c in df.columns]
comentario_col = [c for c in df.columns if "conteudo_comentario" in c][0]
nota_col = [c for c in df.columns if "nota_comentario" in c][0]

df = df[[comentario_col, nota_col]].dropna()
df[nota_col] = pd.to_numeric(df[nota_col].astype(str).str.replace(",", ".", regex=False), errors="coerce").round().clip(1,5)
df[comentario_col] = df[comentario_col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

X_train, X_test, y_train, y_test = train_test_split(
    df[comentario_col], df[nota_col].astype(int),
    test_size=0.2, random_state=42, stratify=df[nota_col]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1,2))),
    ("lr", LogisticRegression(max_iter=200, solver="lbfgs", multi_class="multinomial", class_weight="balanced"))
])

model.fit(X_train, y_train)
joblib.dump(model, "modelo_logreg.pkl")
print("Modelo Logistic Regression salvo: modelo_logreg.pkl")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


✅ Modelo Logistic Regression salvo: modelo_logreg.pkl


#### Métricas e Teste Manual

In [15]:
import joblib
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score

model = joblib.load("modelo_logreg.pkl")

y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
print(f"[LogisticRegression] RMSE={rmse:.3f}  MAE={mae:.3f}  ACC={acc:.3f}")

def predict_one_lr(text: str):
    proba = model.predict_proba([text])[0]
    idx = int(np.argmax(proba))
    classes = model.classes_
    return classes[idx], float(proba[idx]), dict(zip(classes, proba.tolist()))

texts = [
    "Gostei muito do filme, atuação excelente e trilha sonora incrível!",
    "Roteiro fraco e previsível, me decepcionou bastante.",
    "É ok. Nada impressionante, mas também não é ruim."
]
for t in texts:
    label, conf, probs = predict_one_lr(t)
    print(f"{t}\n -> classe: {label} | conf: {conf:.3f} | probs: {probs}\n")

print(df["nota_comentario"].value_counts(normalize=True))

[LogisticRegression] RMSE=1.132  MAE=0.609  ACC=0.585
Gostei muito do filme, atuação excelente e trilha sonora incrível!
 -> classe: 5 | conf: 0.379 | probs: {np.int64(1): 0.11426229954929698, np.int64(2): 0.13358779428224285, np.int64(3): 0.1200940308255452, np.int64(4): 0.2532101509026332, np.int64(5): 0.3788457244402818}

Roteiro fraco e previsível, me decepcionou bastante.
 -> classe: 2 | conf: 0.349 | probs: {np.int64(1): 0.1635105962684895, np.int64(2): 0.3494330716926047, np.int64(3): 0.178859871321098, np.int64(4): 0.14982949881296198, np.int64(5): 0.1583669619048458}

É ok. Nada impressionante, mas também não é ruim.
 -> classe: 2 | conf: 0.362 | probs: {np.int64(1): 0.26367260518805763, np.int64(2): 0.36185843853432087, np.int64(3): 0.18309827649866772, np.int64(4): 0.0917237208902199, np.int64(5): 0.09964695888873376}

nota_comentario
5.0    0.368908
4.0    0.331277
2.0    0.117829
1.0    0.112893
3.0    0.069093
Name: proportion, dtype: float64


### Naive Bayes

#### Treinando e Salvando Modelo

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
import joblib

df = pd.read_csv("comentarios.csv")

df.columns = [c.strip().lower() for c in df.columns]
comentario_col = [c for c in df.columns if "conteudo_comentario" in c][0]
nota_col = [c for c in df.columns if "nota_comentario" in c][0]

df = df[[comentario_col, nota_col]].dropna()
df[nota_col] = pd.to_numeric(df[nota_col].astype(str).str.replace(",", ".", regex=False), errors="coerce").round().clip(1,5)
df[comentario_col] = df[comentario_col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

X_train, X_test, y_train, y_test = train_test_split(
    df[comentario_col], df[nota_col].astype(int),
    test_size=0.2, random_state=42, stratify=df[nota_col]
)

tfidf_union = FeatureUnion([
    ("word", TfidfVectorizer(strip_accents="unicode", lowercase=True)),
    ("char", TfidfVectorizer(analyzer="char_wb"))
])

pipe_nb = Pipeline([
    ("tfidf", tfidf_union),
    ("nb", MultinomialNB())
])

param_grid = {
    "tfidf__word__ngram_range": [(1,1), (1,2)],
    "tfidf__word__min_df": [1,2],
    "tfidf__char__ngram_range": [(3,5), (3,6)],
    "tfidf__char__min_df": [1,2],
    "nb__alpha": [0.1, 0.3, 0.5, 1.0]
}

grid = GridSearchCV(
    pipe_nb,
    param_grid,
    cv=3,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1
)
grid.fit(X_train, y_train)

joblib.dump(model, "modelo_naivebayes.pkl")
print("Modelo Naive Bayes salvo: modelo_naivebayes.pkl")

Fitting 3 folds for each of 64 candidates, totalling 192 fits
Modelo Naive Bayes salvo: modelo_naivebayes.pkl


#### Métricas e Teste Manual

In [24]:
import joblib
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score

model = joblib.load("modelo_naivebayes.pkl")

y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
print(f"[NaiveBayes] RMSE={rmse:.3f}  MAE={mae:.3f}  ACC={acc:.3f}")

def predict_one_nb(text: str):
    proba = model.predict_proba([text])[0]
    idx = int(np.argmax(proba))
    classes = model.classes_
    return classes[idx], float(proba[idx]), dict(zip(classes, proba.tolist()))

texts = [
    "Gostei muito do filme, atuação excelente e trilha sonora incrível!",
    "Roteiro fraco e previsível, me decepcionou bastante.",
    "É ok. Nada impressionante, mas também não é ruim."
]
for t in texts:
    label, conf, probs = predict_one_nb(t)
    print(f"{t}\n -> classe: {label} | conf: {conf:.3f} | probs: {probs}\n")

[NaiveBayes] RMSE=2.966  MAE=1.102  ACC=0.446
Gostei muito do filme, atuação excelente e trilha sonora incrível!
 -> classe: 5 | conf: 0.584 | probs: {np.int64(1): 0.015123000168875852, np.int64(2): 0.020099567245547692, np.int64(3): 0.007590876994897284, np.int64(4): 0.37367187433539345, np.int64(5): 0.5835146812552837}

Roteiro fraco e previsível, me decepcionou bastante.
 -> classe: 4 | conf: 0.418 | probs: {np.int64(1): 0.08085076662326907, np.int64(2): 0.17848977076595868, np.int64(3): 0.049573640257632307, np.int64(4): 0.4175145612044929, np.int64(5): 0.273571261148647}

É ok. Nada impressionante, mas também não é ruim.
 -> classe: 5 | conf: 0.417 | probs: {np.int64(1): 0.08348865656428348, np.int64(2): 0.16937095732215984, np.int64(3): 0.04080520936438455, np.int64(4): 0.2888662177166855, np.int64(5): 0.417468959032486}



## Redes Neurais

### BERT

#### Carregando Dados

In [1]:
# espera df['comentario'], df['nota']

import pandas as pd

df = pd.read_csv("comentarios.csv")
df = pd.DataFrame({
    "comentario": df["conteudo_comentario"].astype(str).str.replace(r"\s+"," ",regex=True).str.strip(),
    "nota": pd.to_numeric(df["nota_comentario"].astype(str).str.replace(",",".",regex=False), errors="coerce")
})
df["nota"] = df["nota"].round().clip(1,5)

#### Treinando Rede Neural - BERT

In [3]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

model_name = "neuralmind/bert-base-portuguese-cased"
max_len = 256
batch_size = 16
epochs = 3
lr = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

class TextRegDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = np.array(labels, dtype=np.float32)
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i],
            truncation=True, padding="max_length", max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item["label"] = torch.tensor(self.labels[i], dtype=torch.float32)
        return item

X_train, X_test, y_train, y_test = train_test_split(
    df["comentario"], df["nota"], test_size=0.2, random_state=42, stratify=df["nota"]
)
train_ds = TextRegDataset(X_train, y_train, tokenizer, max_len)
test_ds  = TextRegDataset(X_test,  y_test,  tokenizer, max_len)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

class BERTRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1)
        masked = last_hidden * mask
        pooled = masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        x = self.dropout(pooled)
        return self.head(x).squeeze(-1)

model = BERTRegressor(model_name).to(device)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
loss_fn = nn.MSELoss()

for epoch in range(epochs):
    model.train()
    running = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attn      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device).float()

        preds = model(input_ids=input_ids, attention_mask=attn)
        loss = loss_fn(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        running += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running/len(train_loader):.4f}")

model.eval()
preds, trues = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attn      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device).float()

        out = model(input_ids=input_ids, attention_mask=attn)
        preds.extend(out.detach().cpu().numpy().tolist())
        trues.extend(labels.detach().cpu().numpy().tolist())

rmse = mean_squared_error(trues, preds)
mae  = mean_absolute_error(trues, preds)
print(f"[BERT] RMSE={rmse:.3f}  MAE={mae:.3f}")


Epoch 1/3 - Loss: 4.1505
Epoch 2/3 - Loss: 0.7126
Epoch 3/3 - Loss: 0.4826
[BERT] RMSE=0.670  MAE=0.588


#### Salvando Modelo

In [4]:
torch.save(model.state_dict(), "bert_regressor.pt")
print("Modelo salvo em: bert_regressor.pt")

Modelo salvo em: bert_regressor.pt


#### Carregando Modelo e Testando

In [8]:
model = BERTRegressor(model_name).to(device)
model.load_state_dict(torch.load("bert_regressor.pt", map_location=device))
model.eval()

def predict_one_bert(text: str, max_length: int = 256):
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=max_length, return_tensors="pt")
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device))
    val = float(out.squeeze().item())
    stars = int(np.clip(np.rint(val), 1, 5))
    return val, stars

raw, s = predict_one_bert("Até que eu gostei do filme")
print(raw, s)

3.6736326217651367 4
